# Tootlus, risk ja portfell

Eestikeelne kursusepraktikum. Käivita ülalt alla. Kõik andmed on simuleeritud või fiktiivsed.

In [ ]:
# Kui paketid puuduvad, käivita see lahter. Colabis võib vaja minna runtime taaskäivitust.
%pip -q install "numpy>=1.26,<3" "pandas>=2.2,<4" "matplotlib>=3.8,<4" "scikit-learn>=1.4,<2"

In [ ]:
import sys, tempfile
from pathlib import Path
course=Path(tempfile.mkdtemp(prefix="investeerimine_"))
(course/"kood").mkdir()
(course/"valjund").mkdir()
(course/"kood"/'tuum.py').write_text('"""Kursuse deterministlik arvutustuumik. Kõik demoandmed on simuleeritud."""\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nROOT = Path(__file__).resolve().parents[1]\nOUT = ROOT / \'valjund\'\nOUT.mkdir(exist_ok=True)\n\ndef demo_hinnad(n=756, seed=42):\n    rng = np.random.default_rng(seed)\n    market = rng.normal(0.00025, 0.009, n)\n    innovations = rng.normal(0, [0.008, 0.010, 0.003], (n, 3))\n    returns = market[:, None] * [1.1, 1.2, 0.2] + innovations\n    prices = 100 * np.cumprod(1 + returns, axis=0)\n    # Esimene rida on investeeringu algväärtus; hilisemad read on vaatlused.\n    prices = np.vstack([np.full(3, 100.), prices])\n    return pd.DataFrame(prices, index=pd.bdate_range(\'2022-01-03\', periods=n+1),\n                        columns=[\'AKTSIA_A_SIM\', \'AKTSIA_B_SIM\', \'VOLAKIRI_SIM\'])\n\ndef kontrolli_hindu(p):\n    if p.empty or len(p) < 2 or not p.index.is_unique or not p.index.is_monotonic_increasing:\n        raise ValueError(\'Vaja on vähemalt kaht järjestatud, kordusteta kuupäeva.\')\n    if p.isna().any().any() or not np.isfinite(p.to_numpy()).all() or (p <= 0).any().any():\n        raise ValueError(\'Puuduv, lõpmatu või mittepositiivne hind. Paranda lähteandmed.\')\n    return p\n\ndef tootlused(p):\n    kontrolli_hindu(p)\n    return p.pct_change(fill_method=None).iloc[1:]\n\ndef moodikud(r, periods=252):\n    a = np.asarray(r, dtype=float)\n    if len(a) < 2 or not np.isfinite(a).all() or (a <= -1).any():\n        raise ValueError(\'Vigased tootlused; see funktsioon eeldab positiivset vara väärtust.\')\n    wealth = np.r_[1., np.cumprod(1+a)]\n    drawdown = wealth / np.maximum.accumulate(wealth) - 1\n    return {\'kogutootlus\': float(wealth[-1]-1),\n            \'CAGR\': float(wealth[-1]**(periods/len(a))-1),\n            \'volatiilsus\': float(np.std(a, ddof=1)*np.sqrt(periods)),\n            \'max_langus\': float(drawdown.min())}\n\ndef osta_hoia(p, weights):\n    kontrolli_hindu(p)\n    w = np.asarray(weights)\n    if len(w)!=len(p.columns) or (w<0).any() or not np.isclose(w.sum(),1):\n        raise ValueError(\'Kaalud peavad olema mittenegatiivsed ja andma summaks 1.\')\n    value = (p / p.iloc[0]).mul(w, axis=1).sum(axis=1)\n    return value.pct_change(fill_method=None).iloc[1:]\n\ndef bond_price(face=1000, coupon=50, years=3, y=0.06):\n    if years < 1 or int(years)!=years or y <= -1:\n        raise ValueError(\'Aastane kupong, täisarv aastaid ning y > -1.\')\n    return sum(coupon/(1+y)**t for t in range(1, years+1)) + face/(1+y)**years\n\ndef ytm(price, face=1000, coupon=50, years=3):\n    lo, hi = -0.9, 10.0\n    if not bond_price(face,coupon,years,hi) <= price <= bond_price(face,coupon,years,lo):\n        raise ValueError(\'Hind väljaspool otsinguvahemikku.\')\n    for _ in range(120):\n        mid=(lo+hi)/2\n        if bond_price(face,coupon,years,mid)>price: lo=mid\n        else: hi=mid\n    return (lo+hi)/2\n\ndef indikaatorid(close):\n    d=close.diff()\n    # Lihtne 14 perioodi RSI; teadlikult mitte Wilderi silumine.\n    gain=d.clip(lower=0).rolling(14).mean()\n    loss=(-d.clip(upper=0)).rolling(14).mean()\n    rsi=100-100/(1+gain/loss.replace(0,np.nan))\n    rsi=rsi.mask((loss==0)&(gain>0),100).mask((loss==0)&(gain==0),50)\n    return pd.DataFrame({\'hind\':close,\'SMA20\':close.rolling(20).mean(),\n                         \'SMA60\':close.rolling(60).mean(),\'RSI14_lihtne\':rsi})\n\ndef backtest(close, cost=0.001):\n    if not 0 <= cost < 0.1: raise ValueError(\'Vigane tehingukulu.\')\n    ind=indikaatorid(close)\n    signal=(ind.SMA20>ind.SMA60).astype(float)\n    # Otsus sulgemise järel t; täitmine järgmise päeva t+1 sulgemisel.\n    # Positsioon teenib tootlust alates t+1 -> t+2. Kulu t+1 päeval.\n    position=signal.shift(2).fillna(0)\n    target_at_close=signal.shift(1).fillna(0)\n    turnover=target_at_close.diff().abs().fillna(target_at_close.iloc[0])\n    r=close.pct_change(fill_method=None).fillna(0)\n    net=(1+position*r)*(1-cost*turnover)-1\n    # Lõpus sulgeme alles oleva positsiooni ja võtame väljumiskulu.\n    net.iloc[-1]=(1+net.iloc[-1])*(1-cost*target_at_close.iloc[-1])-1\n    return pd.DataFrame({\'vara\':r,\'signaal\':signal,\'positsioon\':position,\n                         \'kaive\':turnover,\'neto\':net}).iloc[1:]\n\ndef ml_andmed(close):\n    r=close.pct_change(fill_method=None)\n    X=pd.DataFrame({\'r1\':r,\'r5\':close.pct_change(5,fill_method=None),\n                    \'vol20\':r.rolling(20).std(),\n                    \'kaugus20\':close/close.rolling(20).mean()-1})\n    return X.assign(target=r.shift(-1)).dropna()\n\ndef ml_vordlus(close):\n    from sklearn.ensemble import RandomForestRegressor\n    from sklearn.linear_model import Ridge\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    from sklearn.dummy import DummyRegressor\n    from sklearn.metrics import mean_squared_error\n    from sklearn.model_selection import TimeSeriesSplit\n    d=ml_andmed(close); X=d.drop(columns=\'target\'); y=d.target\n    # Viimane 20% on puutumatu lõputest; üks rida jäetakse piiril välja.\n    cut=int(len(d)*0.8); Xdev=X.iloc[:cut-1]; ydev=y.iloc[:cut-1]\n    models={\'null\':DummyRegressor(strategy=\'constant\',constant=0),\n            \'Ridge\':make_pipeline(StandardScaler(),Ridge(alpha=10)),\n            \'RandomForest\':RandomForestRegressor(n_estimators=80,max_depth=3,\n                                  min_samples_leaf=20,random_state=42,n_jobs=1)}\n    rows=[]\n    for name,model in models.items():\n        cv=[]\n        for tr,va in TimeSeriesSplit(n_splits=3,gap=1).split(Xdev):\n            model.fit(Xdev.iloc[tr],ydev.iloc[tr])\n            cv.append(np.sqrt(mean_squared_error(ydev.iloc[va],model.predict(Xdev.iloc[va]))))\n        model.fit(Xdev,ydev)\n        pred=model.predict(X.iloc[cut:])\n        rows.append({\'mudel\':name,\'CV_RMSE\':np.mean(cv),\n                     \'test_RMSE\':np.sqrt(mean_squared_error(y.iloc[cut:],pred))})\n    return pd.DataFrame(rows), {\'treening_lopp\':str(Xdev.index[-1]),\n        \'test_algus\':str(X.index[cut]),\'test_ridu\':len(X)-cut}\n',encoding="utf-8")
sys.path.insert(0,str(course/"kood"))
print("Ajutine töökaust:",course)


## 1. Simuleeritud andmed
Andmed ei ole USA turu ajalugu. Vaata esimesi ridu ja nimeta hindade ühikud.

In [ ]:
from tuum import demo_hinnad,tootlused,moodikud,osta_hoia
p=demo_hinnad()
print(p.head())

## 2. Risk ja kasv
Ennusta, kas hajutamine vähendab riski. Kaalud on algkaalud.

In [ ]:
import pandas as pd
r=tootlused(p)
print(pd.DataFrame({c:moodikud(r[c]) for c in p}).T)
print(moodikud(osta_hoia(p,[1/3]*3)))

In [ ]:
import matplotlib.pyplot as plt
(p/p.iloc[0]).plot(title="Simuleeritud väärtus, algus = 1")
plt.show()

## Iseseisev töö
Muuda kaalud 60%, 20%, 20%. Selgita, kuidas muutus tootlus ja maksimaalne langus. Seejärel kontrolli kahe vara kaalude triivi käsitsi; lahendus on kursuse LAHENDUSED.md M4 all.